In [ ]:
!pip install --upgrade --force-reinstall sympy torch torchvision

In [ ]:
!pip install --force-reinstall --no-cache-dir numpy

In [ ]:
# Completely wipe any conflicting versions
# !pip uninstall -y torch torchvision torchtext

# Reinstall the latest versions directly from PyTorch's official source with modern CUDA support
# !pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

from glob import glob
from PIL import Image
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

path = kagglehub.dataset_download("nih-chest-xrays/data")

print("Path to dataset files:", path)

# for filename in os.listdir(path):
#     print(filename)

# print(os.getcwd())
print(path + "/Data_Entry_2017.csv")

df = pd.read_csv(path + "/Data_Entry_2017.csv", nrows=2000)

# Read images
target_folder = os.path.join(path, "images_001", "images", "*.png")
all_image_paths = glob(target_folder)
image_path_dict = {os.path.basename(p): p for p in all_image_paths}
print(f"Indexed {len(image_path_dict)} images from images_001")

df = df[df["Image Index"].isin(image_path_dict.keys())].reset_index(drop=True)
print(f"Rows remaining after filtering for images_001: {len(df)}")

X = df[['Image Index']]
y = df['Finding Labels']

# Pass the Patient_id column so the splitter keeps patients intact
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(X, y, groups=df['Patient ID']):
    train_set = df.iloc[train_idx]
    test_set = df.iloc[test_idx]

# print("Train set: \n", train_set)
# print("Test set: \n",  test_set)

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_path_dict, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_path_dict = image_path_dict
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["Image Index"]

        img_path = self.image_path_dict.get(img_name)
        if img_path is None:
            raise FileNotFoundError(
                f"Could not find file for image index: {img_name}"
            )

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label_str = row["Finding Labels"]
        label = 0.0 if "No Finding" in label_str else 1.0
        label_tensor = torch.tensor(label, dtype=torch.float32)

        return image, label_tensor



# 8. Define standard CNN image preprocessing transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

train_dataset = ChestXrayDataset(train_set, image_path_dict, transform=transform)
test_dataset = ChestXrayDataset(test_set, image_path_dict, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 11. Sanity check a single batch
images, labels = next(iter(train_loader))
print("Batch image tensor shape:", images.shape)  # Expected: torch.Size([32, 3, 224, 224])
print("Batch labels (first 5):", labels[:5])

Path to dataset files: /kaggle/input/datasets/nih-chest-xrays/data
/kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv
Indexed 4999 images from images_001
Rows remaining after filtering for images_001: 2000
Batch image tensor shape: torch.Size([32, 3, 224, 224])
Batch labels (first 5): tensor([0., 0., 0., 1., 0.])


In [ ]:
# Code for the Conv network and training

# 11. Define Neural Network (2 Convolutional Layers + 1 ANN Classifier)
class SimpleCNN(nn.Module):

  def __init__(self):
    super(SimpleCNN, self).__init__()
    # Convolutional Layer 1
    self.conv1 = nn.Conv2d(
        in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1
    )
    # Convolutional Layer 2
    self.conv2 = nn.Conv2d(
        in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1
    )
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    # ANN / Fully Connected Layer (Input dimensions: 32 channels * 56 * 56 features)
    self.fc1 = nn.Linear(32 * 56 * 56, 1)

  def forward(self, x):
    # Pass through Conv Layer 1 -> ReLU -> MaxPool
    x = self.pool(F.relu(self.conv1(x)))
    # Pass through Conv Layer 2 -> ReLU -> MaxPool
    x = self.pool(F.relu(self.conv2(x)))

    # Flatten feature maps into a 1D vector for the ANN layer
    x = x.view(x.size(0), -1)

    # Pass through ANN linear layer with a Sigmoid activation for binary output
    x = torch.sigmoid(self.fc1(x))
    return x


# 12. Initialize Model, Loss Function, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SimpleCNN().to(device)
criterion = (
    nn.BCELoss()
)  # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Resnet-18 model with pretrained weights
import torchvision.models as models


# 11. Define Neural Network using ResNet18
class ResNet18Binary(nn.Module):

  def __init__(self, pretrained=True):
    super(ResNet18Binary, self).__init__()
    # Load ResNet18 (using weights='DEFAULT' or pretrained=True depending on torchvision version)
    try:
      weights = None #models.ResNet18_Weights.DEFAULT if pretrained else None
      self.resnet = models.resnet18(weights=weights)
    except AttributeError:
      # Fallback for older torchvision versions
      self.resnet = models.resnet18(pretrained=pretrained)

    # ResNet18 outputs 512 features from its final average pooling layer before the fc layer.
    # Replace the default 1000-class classifier with a 1-output linear layer for binary classification.
    num_features = self.resnet.fc.in_features
    self.resnet.fc = nn.Linear(num_features, 1)

  def forward(self, x):
    # Pass through ResNet layers
    x = self.resnet(x)
    # Apply sigmoid activation to get a probability between 0 and 1
    x = torch.sigmoid(x)
    return x

# 12. Initialize Model, Loss Function, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize ResNet18 binary model
model = ResNet18Binary(pretrained=True).to(device)

# to train Resnet-18 model from scratch
# model = ResNet18Binary(pretrained=False).to(device)

criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(
    model.parameters(), lr=0.0001
)  # Lower learning rate (e.g., 1e-4) is often better for pre-trained weights

# 13. Training Loop
num_epochs = 15
print("Starting training loop...")

for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0

  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    # Zero out gradients from previous step
    optimizer.zero_grad()

    # Forward pass
    outputs = model(images).squeeze()

    # Compute loss
    loss = criterion(outputs, labels)

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

  epoch_loss = running_loss / len(train_dataset)
  print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss:.4f}")



print("Training complete!")

Using device: cuda
Using device: cuda
Starting training loop...
Epoch [1/15], Training Loss: 0.6808
Epoch [2/15], Training Loss: 0.6267
Epoch [3/15], Training Loss: 0.5224
Epoch [4/15], Training Loss: 0.4255
Epoch [5/15], Training Loss: 0.2906
Epoch [6/15], Training Loss: 0.1717
Epoch [7/15], Training Loss: 0.0967
Epoch [8/15], Training Loss: 0.0929
Epoch [9/15], Training Loss: 0.0631


In [3]:
# 14. Testing and Evaluation Phase
model.eval()  # Set model to evaluation mode (turns off dropout/batchnorm updates if any)
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():  # Disable gradient calculations during evaluation
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(images).squeeze()

    # Compute loss
    loss = criterion(outputs, labels)
    test_loss += loss.item() * images.size(0)

    # Calculate accuracy (threshold outputs at 0.5 for binary classification)
    predicted = (outputs >= 0.5).float()
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

# Compute final average loss and accuracy
avg_test_loss = test_loss / len(test_dataset)
test_accuracy = correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(
    f"Test Accuracy: {test_accuracy * 100:.2f}% ({correct}/{total} correct)"
)

Test Loss: 0.7881
Test Accuracy: 60.93% (92/151 correct)
